# DiveSensei R41 PaliGemma Remote GPU Benchmark - Kaggle

Research-only visual proposal probe. This does not change `approve_review_v1`, taxonomy, auto-approval, or auto-exclusion.

Before running: enable a GPU accelerator, attach the private Dataset `maximecauchy/divesensei-r41-remote-gpu-package-v3`, and add a Kaggle secret named `HF_TOKEN` with a Hugging Face token that has accepted access to `google/paligemma2-3b-mix-224`.


In [ ]:
import os, shutil, subprocess, sys, tarfile
from pathlib import Path

print('Python', sys.version)
subprocess.run(['nvidia-smi'], check=False)

import torch
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is not enabled. Enable Accelerator > GPU before running this notebook.')
print('cuda_device_count', torch.cuda.device_count())
for idx in range(torch.cuda.device_count()):
    print('gpu', idx, torch.cuda.get_device_name(idx), torch.cuda.get_device_capability(idx))


In [ ]:
# Install only the remote benchmark dependencies. Keep this cell deterministic.
!python -m pip install -q --upgrade pip
!python -m pip install -q torch transformers==4.53.3 accelerate sentencepiece huggingface-hub opencv-python-headless Pillow


In [ ]:
# Resolve Hugging Face token from Kaggle Secrets.
import time

token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
if token:
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGINGFACE_HUB_TOKEN'] = token
    print('HF_TOKEN already present in environment')
else:
    for attempt in range(3):
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret('HF_TOKEN')
            if token:
                os.environ['HF_TOKEN'] = token
                os.environ['HUGGINGFACE_HUB_TOKEN'] = token
                print('HF_TOKEN loaded from Kaggle secret')
                break
            print('HF_TOKEN secret is empty')
            break
        except Exception as exc:
            print(f'Could not read Kaggle HF_TOKEN secret (attempt {attempt + 1}/3):', repr(exc))
            if attempt == 2:
                break
            time.sleep(5)


In [ ]:
# Resolve attached dataset package directory directly from Kaggle Input.
PACKAGE_CANDIDATES = [
    Path('/kaggle/input/r41-remote-gpu-package-v3/r41_remote_gpu_package'),
    Path('/kaggle/input/r41-remote-gpu-package-v3'),
    Path('/kaggle/input/divesensei-r41-remote-gpu-package-v3/r41_remote_gpu_package'),
    Path('/kaggle/input/divesensei-r41-remote-gpu-package-v3'),
]
package_root = next(
    (
        p
        for p in [*PACKAGE_CANDIDATES, *Path('/kaggle/input').glob('**/r41_remote_gpu_package'), *Path('/kaggle/input').glob('**/REMOTE_PACKAGE_MANIFEST.json')]
        if ((p / 'REMOTE_PACKAGE_MANIFEST.json').exists() if p.is_dir() else p.name == 'REMOTE_PACKAGE_MANIFEST.json')
    ),
    None,
)
if package_root is not None and package_root.is_file():
    package_root = package_root.parent
if package_root is None:
    raise FileNotFoundError('Attach the private Kaggle Dataset maximecauchy/r41-remote-gpu-package-v3 first.')

work_root = Path('/kaggle/working')
print('Package root:', package_root)
print((package_root / 'REMOTE_PACKAGE_MANIFEST.json').read_text()[:1000])


In [ ]:
# Preflight + one-frame smoke + audio-gated center-pool 1 FPS benchmark.
# The runner rejects CPU-only execution by default.
os.environ['HF_HOME'] = '/kaggle/working/hf-cache'
cmd = [
    sys.executable, 'benchmarks/r41_remote_gpu_runner.py',
    '--cache-dir', os.environ['HF_HOME'],
    '--output-root', '/kaggle/working/r41_remote_gpu_results_center_pool',
    '--prompt-id', 'diving_attempt',
    '--decision-rule', 'yes_no_first_token_margin',
    '--roi-mode', 'center_pool',
    '--smoke-max-frames', '1',
    '--full-fps', '1.0',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=package_root, check=True)


In [ ]:
# Inspect and expose result bundle for download from Kaggle Output.
summary = Path('/kaggle/working/r41_remote_gpu_results_center_pool/r41_remote_gpu_run_summary.md')
print(summary.read_text() if summary.exists() else 'missing summary')
print('Bundle:', Path('/kaggle/working/r41_remote_gpu_results_bundle.zip'), Path('/kaggle/working/r41_remote_gpu_results_bundle.zip').exists())
